# NB12 — Two-Stage Architecture with K-Fold OOF Cross-Fitting (1h, BTC)

NB10's 50/50 chronological split of the training slice halved the Stage-2 sample size and dropped `baseline_rf` IC from NB08/09's +0.027 to NB10's +0.020 — a +0.007 IC haircut purely from the split. NB12 replaces that split with **time-respecting K-fold OOF cross-fitting**:

Within each walk-forward fold's train slice:

1. Run `TimeSeriesSplit(n_splits=4)` to get 4 expanding-window inner folds.
2. For each inner fold `j`, train Stage-1 `f1` on inner-train, predict `P(z_{t+1h})` on inner-test. The first inner-fold has no Stage-1 model (no prior data); its predictions are filled with uniform `1/K`.
3. Concatenate inner-test predictions across all 4 inner folds → OOF transition predictions for the **entire training slice**.
4. Train a final Stage-1 model on the full training slice (used to predict on the walk-forward test slice).
5. Stage 2 trains on the **full training slice** using OOF transition features. Test-set transition features come from the full-train-fit Stage-1.

This recovers ~50–75 % of the Stage-2 training data (depending on `n_splits`) and gives the transition features a fair test against `hierarchy_rf` trained on the same data.

**Question:** does the +0.0005 IC lift NB10 saw at 1h (and +0.0104 NB11 saw at the same horizon with shifted folds) grow when Stage 2 has the full training slice? If yes, NB10's null was partly a sample-size artifact. If still tiny, the hypothesis is genuinely scope-limited to longer horizons (NB11 4h is the cleaner instance).


In [1]:
import math, os, json, random
from dataclasses import dataclass
from datetime import datetime
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    accuracy_score, balanced_accuracy_score, roc_auc_score,
)
from scipy.stats import spearmanr

SEED = 42
random.seed(SEED); np.random.seed(SEED)

TICKER = "BTC-USD"
PERIOD = "720d"
INTERVAL = "1h"

HORIZON = 1
N_FOLDS = 5
MIN_TRAIN_FRACTION = 0.40
TEST_FRACTION_PER_FOLD = 0.10
INNER_N_SPLITS = 4           # TimeSeriesSplit for OOF cross-fitting within each train slice

MAX_HIERARCHY_DEPTH = 2
MIN_LEAF_COUNT = 200

RF_N_ESTIMATORS = 200
STAGE1_PARAMS = {"max_depth": 8, "min_samples_leaf": 20}
STAGE2_PARAMS = {"max_depth": 8, "min_samples_leaf": 20}

JOURNAL_PATH = "EXPERIMENTS.md"


## Reused helpers (identical to NB10/NB11)


In [2]:
def load_yfinance_hourly(ticker, period, interval):
    raw = yf.download(tickers=ticker, period=period, interval=interval,
                     auto_adjust=True, progress=False, group_by="column")
    if raw.empty: raise RuntimeError(f"No data for {ticker}")
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = [c[0].lower() for c in raw.columns]
    else:
        raw.columns = [str(c).lower() for c in raw.columns]
    df = raw[["open","high","low","close","volume"]].copy().reset_index()
    ts_col = [c for c in df.columns if c not in ("open","high","low","close","volume")][0]
    df = df.rename(columns={ts_col: "ts"})
    df["ts"] = pd.to_datetime(df["ts"], utc=True)
    return df.dropna().sort_values("ts").reset_index(drop=True)


def rolling_zscore(x, w, eps=1e-8):
    m = x.rolling(w, min_periods=max(3, w // 3)).mean()
    s = x.rolling(w, min_periods=max(3, w // 3)).std()
    return (x - m) / (s + eps)


def sigmoid_np(x): return 1.0 / (1.0 + np.exp(-x))


def make_hourly_features(df, horizon):
    out = df.copy()
    for lag in (1, 2, 4, 12, 24, 168):
        out[f"ret_{lag}"] = np.log(out["close"] / out["close"].shift(lag))
    out["range_pct"] = (out["high"] - out["low"]) / out["close"]
    out["open_close_pct"] = (out["close"] - out["open"]) / out["open"]
    for w in (24, 168, 720):
        out[f"vol_{w}"] = out["ret_1"].rolling(w).std()
    for w in (24, 168, 720):
        out[f"ma_{w}"] = out["close"].rolling(w).mean()
        out[f"price_to_ma_{w}"] = out["close"] / out[f"ma_{w}"] - 1.0
    out["volume_log"] = np.log1p(out["volume"])
    out["volume_z_168"] = rolling_zscore(out["volume_log"], 168)
    for w in (168, 720):
        out[f"rolling_max_{w}"] = out["close"].rolling(w).max()
        out[f"drawdown_{w}"] = out["close"] / out[f"rolling_max_{w}"] - 1.0
    out["trend_pressure"] = rolling_zscore(out["ret_168"], 720)
    out["fear_pressure"] = (-4.0*out["drawdown_168"] - 2.0*out["drawdown_720"]
                              + 8.0*out["vol_168"] + 0.35*out["volume_z_168"]
                              - 0.50*out["ret_24"])
    out["greed_pressure"] = (1.50*out["trend_pressure"] + 2.00*out["price_to_ma_168"]
                              + 1.00*out["ret_168"] + 0.20*out["volume_z_168"]
                              - 3.00*out["vol_720"])
    out["fear_index"] = sigmoid_np(out["fear_pressure"].clip(-8, 8))
    out["greed_index"] = sigmoid_np(out["greed_pressure"].clip(-8, 8))
    out["future_log_return"] = np.log(out["close"].shift(-horizon) / out["close"])
    out["future_direction"] = (out["future_log_return"] > 0).astype(int)
    return out.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)


BASELINE_FEATURES = [
    "ret_1", "ret_2", "ret_4", "ret_12", "ret_24", "ret_168",
    "range_pct", "open_close_pct",
    "vol_24", "vol_168", "vol_720",
    "price_to_ma_24", "price_to_ma_168", "price_to_ma_720",
    "volume_z_168", "drawdown_168", "drawdown_720",
]


@dataclass
class MeasureNode:
    node_id: str; depth: int
    bounds: Dict[str, Tuple[float, float]]
    mass: float; count: int
    mean: Dict[str, float]
    children: Optional[List["MeasureNode"]] = None
    @property
    def is_leaf(self): return not self.children


def in_bounds(row, bounds):
    for col, (lo, hi) in bounds.items():
        v = float(row[col])
        if not (lo <= v < hi): return False
    return True


def split_node(n, sc, sv):
    lo, hi = n.bounds[sc]
    lb, rb = dict(n.bounds), dict(n.bounds)
    lb[sc] = (lo, sv); rb[sc] = (sv, hi)
    n.children = [
        MeasureNode(n.node_id+"L", n.depth+1, lb, n.mass/2, 0, {}, None),
        MeasureNode(n.node_id+"R", n.depth+1, rb, n.mass/2, 0, {}, None),
    ]
    return n.children


def collect_leaves(n):
    if n.is_leaf: return [n]
    return [l for c in n.children for l in collect_leaves(c)]


def build_hierarchy(train_df, max_depth, min_count):
    feat = ["greed_index", "fear_index"]
    root = MeasureNode("root", 0,
                       {"greed_index":(0.0,1.000001), "fear_index":(0.0,1.000001)},
                       1.0, len(train_df),
                       {c: float(train_df[c].mean()) for c in feat}, None)
    queue = [root]
    while queue:
        n = queue.pop(0)
        if n.depth >= max_depth: continue
        mask = train_df.apply(lambda r: in_bounds(r, n.bounds), axis=1)
        sub = train_df[mask]
        n.count = len(sub); n.mass = n.count / max(len(train_df), 1)
        if n.count < min_count: continue
        v = {c: float(sub[c].var()) for c in feat}
        sc = max(v, key=v.get); sv = float(sub[sc].median())
        if n.bounds[sc][0] < sv < n.bounds[sc][1]:
            queue.extend(split_node(n, sc, sv))
    for leaf in collect_leaves(root):
        mask = train_df.apply(lambda r: in_bounds(r, leaf.bounds), axis=1)
        sub = train_df[mask]
        leaf.count = len(sub); leaf.mass = leaf.count / max(len(train_df), 1)
        leaf.mean = {c: (float(sub[c].mean()) if len(sub) else 0.0) for c in feat}
    return root


def soft_membership(row, leaves, temperature=0.04):
    p = np.array([float(row["greed_index"]), float(row["fear_index"])])
    centers = np.array([[(l.bounds["greed_index"][0]+l.bounds["greed_index"][1])/2,
                          (l.bounds["fear_index"][0]+l.bounds["fear_index"][1])/2]
                         for l in leaves])
    d = np.sum((centers - p[None,:])**2, axis=1)
    logits = -d / max(temperature, 1e-8); logits -= logits.max()
    w = np.exp(logits); return w / w.sum()


def add_memberships(frame, leaves, cols):
    M = np.vstack([soft_membership(r, leaves) for _, r in frame.iterrows()])
    return pd.concat([frame.reset_index(drop=True),
                      pd.DataFrame(M, columns=cols).reset_index(drop=True)], axis=1)


def assign_leaf_index(row, leaves):
    for i, leaf in enumerate(leaves):
        if in_bounds(row, leaf.bounds): return i
    return -1


def make_folds(n_rows, n_folds, min_train_fraction, test_fraction_per_fold):
    folds = []
    for i in range(n_folds):
        train_end = int(n_rows * (min_train_fraction + i * test_fraction_per_fold))
        test_end = int(n_rows * (min_train_fraction + (i + 1) * test_fraction_per_fold))
        if test_end > n_rows: test_end = n_rows
        if test_end - train_end < 50: break
        folds.append((train_end, test_end))
    return folds


def ic(y_true, y_pred):
    if np.std(y_pred) == 0: return 0.0
    rho, _ = spearmanr(y_true, y_pred)
    return float(rho) if not np.isnan(rho) else 0.0


def derive_transition_features(probs, state_t, leaf_centers, K):
    eps = 1e-12
    entropy_vals = -np.sum(probs * np.log(probs + eps), axis=1)
    p_same = probs[np.arange(len(probs)), state_t]
    instability = 1.0 - p_same
    state_indices = np.arange(K)
    expected_state_idx = probs @ state_indices
    delta_index = expected_state_idx - state_t.astype(float)
    expected_center = probs @ leaf_centers
    current_center = leaf_centers[state_t]
    drift = expected_center - current_center
    sorted_probs = np.sort(probs, axis=1)[:, ::-1]
    margin = sorted_probs[:, 0] - sorted_probs[:, 1]
    argmax_state = probs.argmax(axis=1)
    state_change = (argmax_state != state_t).astype(int)
    out = {f"trans_prob_{i}": probs[:, i] for i in range(K)}
    out.update({
        "trans_entropy": entropy_vals,
        "trans_instability": instability,
        "trans_delta_index": delta_index,
        "trans_drift_greed": drift[:, 0],
        "trans_drift_fear": drift[:, 1],
        "trans_margin": margin,
        "trans_state_change": state_change,
    })
    return pd.DataFrame(out)


TRANSITION_FEATURE_NAMES_TEMPLATE = (
    lambda K: [f"trans_prob_{i}" for i in range(K)] + [
        "trans_entropy", "trans_instability", "trans_delta_index",
        "trans_drift_greed", "trans_drift_fear", "trans_margin", "trans_state_change",
    ]
)


def _rf_reg(params=STAGE2_PARAMS):
    return RandomForestRegressor(n_estimators=RF_N_ESTIMATORS, random_state=SEED, n_jobs=-1, **params)

def _rf_cls(params=STAGE2_PARAMS):
    return RandomForestClassifier(n_estimators=RF_N_ESTIMATORS, random_state=SEED, n_jobs=-1,
                                   class_weight="balanced_subsample", **params)


## K-fold OOF cross-fitting for Stage-1 predictions

Within each walk-forward fold's `train_m`, we run `TimeSeriesSplit(n_splits=4)` to generate four expanding-window inner folds. For each inner fold `j`, we fit Stage-1 on `inner_train` and predict on `inner_test`. The first inner fold's `inner_train` is small but non-empty (`TimeSeriesSplit` ensures each test set comes *after* its training set, so all inner folds have a usable train). We collect predictions for all `inner_test` rows; the inner-test ranges cover the entire `train_m` after the first warm-up portion, which `TimeSeriesSplit` excludes from any test slice.

Rows in `train_m` that are never in any inner-test slice (the initial warm-up segment) get `1/K` uniform predictions as a fallback. This is a clean fallback — it tells Stage 2 "we have no transition prediction for this row, treat it as maximum uncertainty."


In [3]:
def cross_fit_transition_features(train_m, s1_features, leaf_centers, K,
                                  n_splits=INNER_N_SPLITS, target_col="state_t_future"):
    n = len(train_m)
    oof_probs = np.full((n, K), 1.0 / K)             # uniform fallback for warm-up rows
    covered_mask = np.zeros(n, dtype=bool)

    tss = TimeSeriesSplit(n_splits=n_splits)
    for fold_idx, (inner_train_idx, inner_test_idx) in enumerate(tss.split(train_m)):
        f1 = _rf_cls(STAGE1_PARAMS)
        f1.fit(train_m.iloc[inner_train_idx][s1_features],
               train_m.iloc[inner_train_idx][target_col])
        classes_seen = list(f1.classes_)
        probs_inner = f1.predict_proba(train_m.iloc[inner_test_idx][s1_features])
        full_probs = pd.DataFrame(probs_inner, columns=[f"_c{c}" for c in classes_seen]).reindex(
            columns=[f"_c{c}" for c in range(K)], fill_value=0.0
        ).to_numpy()
        oof_probs[inner_test_idx] = full_probs
        covered_mask[inner_test_idx] = True

    oof_trans = derive_transition_features(
        oof_probs, train_m["state_t"].to_numpy(), leaf_centers, K
    )
    return oof_trans, oof_probs, covered_mask


## Per-fold walk-forward runner


In [4]:
def run_one_fold_oof(df_feat, fold_idx, train_end, test_end):
    train = df_feat.iloc[:train_end].copy()
    test = df_feat.iloc[train_end:test_end].copy()

    root = build_hierarchy(train, MAX_HIERARCHY_DEPTH, MIN_LEAF_COUNT)
    leaves = collect_leaves(root)
    K = len(leaves)
    leaf_cols = [f"leaf_{l.node_id}" for l in leaves]
    leaf_centers = np.array([[(l.bounds["greed_index"][0]+l.bounds["greed_index"][1])/2,
                                (l.bounds["fear_index"][0]+l.bounds["fear_index"][1])/2]
                               for l in leaves])
    train_m = add_memberships(train, leaves, leaf_cols)
    test_m = add_memberships(test, leaves, leaf_cols)

    train_m["state_t"] = train_m.apply(lambda r: assign_leaf_index(r, leaves), axis=1)
    test_m["state_t"] = test_m.apply(lambda r: assign_leaf_index(r, leaves), axis=1)
    train_m["state_t_future"] = train_m["state_t"].shift(-HORIZON)
    train_m = train_m.dropna(subset=["state_t_future"]).reset_index(drop=True)
    train_m["state_t_future"] = train_m["state_t_future"].astype(int)
    train_m["state_t"] = train_m["state_t"].astype(int)
    test_m["state_t"] = test_m["state_t"].astype(int)

    s1_features = BASELINE_FEATURES + leaf_cols

    # OOF cross-fitting for transition features on the training slice
    oof_trans, oof_probs, covered_mask = cross_fit_transition_features(
        train_m, s1_features, leaf_centers, K
    )
    train_m = pd.concat([train_m.reset_index(drop=True), oof_trans.reset_index(drop=True)], axis=1)

    # Final Stage-1 fit on full training slice, applied to test
    f1_full = _rf_cls(STAGE1_PARAMS)
    f1_full.fit(train_m[s1_features], train_m["state_t_future"])
    classes_seen = list(f1_full.classes_)
    test_probs = pd.DataFrame(f1_full.predict_proba(test_m[s1_features]),
                              columns=[f"_c{c}" for c in classes_seen]).reindex(
        columns=[f"_c{c}" for c in range(K)], fill_value=0.0
    ).to_numpy()
    test_trans = derive_transition_features(test_probs, test_m["state_t"].to_numpy(), leaf_centers, K)
    test_m = pd.concat([test_m.reset_index(drop=True), test_trans.reset_index(drop=True)], axis=1)

    trans_cols = TRANSITION_FEATURE_NAMES_TEMPLATE(K)
    feature_sets = [
        ("baseline_rf", BASELINE_FEATURES),
        ("hierarchy_rf", BASELINE_FEATURES + leaf_cols),
        ("transition_rf", BASELINE_FEATURES + leaf_cols + trans_cols),
    ]
    y_r = test_m["future_log_return"].to_numpy()
    y_c = test_m["future_direction"].to_numpy().astype(int)

    rows = []
    for name, fcols in feature_sets:
        reg = _rf_reg(); reg.fit(train_m[fcols], train_m["future_log_return"])
        cls = _rf_cls(); cls.fit(train_m[fcols], train_m["future_direction"].astype(int))
        pred_r = reg.predict(test_m[fcols])
        pred_c = cls.predict(test_m[fcols])
        prob_c = cls.predict_proba(test_m[fcols])[:, 1] if 1 in cls.classes_ else np.full(len(y_c), 0.5)
        rows.append({"fold": fold_idx, "model": name, "kind": "regression",
                     "mae": float(mean_absolute_error(y_r, pred_r)),
                     "rmse": float(math.sqrt(mean_squared_error(y_r, pred_r))),
                     "ic": ic(y_r, pred_r), "n_test": len(y_r)})
        rows.append({"fold": fold_idx, "model": name, "kind": "classification",
                     "accuracy": float(accuracy_score(y_c, pred_c)),
                     "balanced_accuracy": float(balanced_accuracy_score(y_c, pred_c)),
                     "roc_auc": float(roc_auc_score(y_c, prob_c)) if len(np.unique(y_c)) == 2 else float("nan"),
                     "n_test": len(y_c)})

    pr_zero = np.zeros_like(y_r)
    pr_pers = test_m["ret_1"].to_numpy()
    majority = int(train_m["future_direction"].mean() >= 0.5)
    pc_majority = np.full_like(y_c, majority)
    pc_pers = (pr_pers > 0).astype(int)
    for name, pr in [("zero", pr_zero), ("persistence", pr_pers)]:
        rows.append({"fold": fold_idx, "model": name, "kind": "regression",
                     "mae": float(mean_absolute_error(y_r, pr)),
                     "rmse": float(math.sqrt(mean_squared_error(y_r, pr))),
                     "ic": ic(y_r, pr), "n_test": len(y_r)})
    for name, pc in [("majority", pc_majority), ("persistence", pc_pers)]:
        rows.append({"fold": fold_idx, "model": name, "kind": "classification",
                     "accuracy": float(accuracy_score(y_c, pc)),
                     "balanced_accuracy": float(balanced_accuracy_score(y_c, pc)),
                     "roc_auc": float("nan"),
                     "n_test": len(y_c)})

    info = {"K": K, "n_train": len(train_m), "n_test": len(test_m),
            "oof_covered_fraction": float(covered_mask.mean()),
            "f1_full_acc_on_train": float(accuracy_score(train_m["state_t_future"], f1_full.predict(train_m[s1_features]))),
            "f1_persistence_acc_on_train": float((train_m["state_t"] == train_m["state_t_future"]).mean())}
    return pd.DataFrame(rows), info


## Run all folds


In [5]:
df_raw = load_yfinance_hourly(TICKER, PERIOD, INTERVAL)
df_feat = make_hourly_features(df_raw, HORIZON)
print(f"{TICKER}: {len(df_raw)} raw -> {len(df_feat)} bars after dropna")

folds = make_folds(len(df_feat), N_FOLDS, MIN_TRAIN_FRACTION, TEST_FRACTION_PER_FOLD)
all_rows, all_info = [], []
for i, (tr, te) in enumerate(folds):
    print(f"fold={i} train_end={tr} test_end={te}")
    rdf, info = run_one_fold_oof(df_feat, i, tr, te)
    info["fold"] = i
    all_rows.append(rdf)
    all_info.append(info)

agg = pd.concat(all_rows, ignore_index=True)
info_df = pd.DataFrame(all_info)
print()
print("Per-fold info:")
print(info_df.round(4))


BTC-USD: 17101 raw -> 16380 bars after dropna
fold=0 train_end=6552 test_end=8190


fold=1 train_end=8190 test_end=9828


fold=2 train_end=9828 test_end=11466


fold=3 train_end=11466 test_end=13104


fold=4 train_end=13104 test_end=14742



Per-fold info:
   K  n_train  n_test  oof_covered_fraction  f1_full_acc_on_train  \
0  4     6551    1638                0.7999                0.8935   
1  4     8189    1638                0.7996                0.8811   
2  4     9827    1638                0.7998                0.8770   
3  4    11465    1638                0.8000                0.8782   
4  4    13103    1638                0.7998                0.8761   

   f1_persistence_acc_on_train  fold  
0                       0.8809     0  
1                       0.8657     1  
2                       0.8612     2  
3                       0.8618     3  
4                       0.8610     4  


## Per-fold IC + lift summary


In [6]:
reg = agg[agg["kind"] == "regression"]
cls = agg[agg["kind"] == "classification"]
ic_pivot = reg[reg["model"].isin(["baseline_rf", "hierarchy_rf", "transition_rf"])].pivot_table(
    index="fold", columns="model", values="ic"
)
ba_pivot = cls[cls["model"].isin(["baseline_rf", "hierarchy_rf", "transition_rf"])].pivot_table(
    index="fold", columns="model", values="balanced_accuracy"
)
auc_pivot = cls[cls["model"].isin(["baseline_rf", "hierarchy_rf", "transition_rf"])].pivot_table(
    index="fold", columns="model", values="roc_auc"
)
print("=== Per-fold IC (1h, OOF cross-fitting) ===")
print(ic_pivot.round(5))
print()
print("=== Per-fold balanced accuracy ===")
print(ba_pivot.round(4))
print()
print("=== Per-fold ROC AUC ===")
print(auc_pivot.round(4))
print()
print("--- Lifts: transition_rf − hierarchy_rf ---")
print(pd.DataFrame({
    "ic_lift": ic_pivot["transition_rf"] - ic_pivot["hierarchy_rf"],
    "bal_acc_lift_pp": (ba_pivot["transition_rf"] - ba_pivot["hierarchy_rf"]) * 100,
    "auc_lift": auc_pivot["transition_rf"] - auc_pivot["hierarchy_rf"],
}).round(5))
print()
print("--- Mean ± std lifts ---")
ic_lift = ic_pivot["transition_rf"] - ic_pivot["hierarchy_rf"]
ba_lift = ba_pivot["transition_rf"] - ba_pivot["hierarchy_rf"]
auc_lift = auc_pivot["transition_rf"] - auc_pivot["hierarchy_rf"]
print(f"IC lift:   mean = {ic_lift.mean():+.5f}, std = {ic_lift.std():.5f}, folds positive = {int((ic_lift > 0).sum())}/5")
print(f"bal_acc:   mean = {ba_lift.mean()*100:+.3f} pp, std = {ba_lift.std()*100:.3f}, folds positive = {int((ba_lift > 0).sum())}/5")
print(f"AUC lift:  mean = {auc_lift.mean():+.5f}, std = {auc_lift.std():.5f}, folds positive = {int((auc_lift > 0).sum())}/5")


=== Per-fold IC (1h, OOF cross-fitting) ===
model  baseline_rf  hierarchy_rf  transition_rf
fold                                           
0          0.03781       0.05077        0.05996
1         -0.02127      -0.01022       -0.00419
2          0.03693       0.03819        0.03631
3          0.03894       0.05288        0.04522
4          0.03755       0.02715        0.02659

=== Per-fold balanced accuracy ===
model  baseline_rf  hierarchy_rf  transition_rf
fold                                           
0           0.5121        0.5260         0.5261
1           0.4899        0.5141         0.5136
2           0.5150        0.5104         0.5280
3           0.5178        0.5128         0.5250
4           0.5306        0.5128         0.5092

=== Per-fold ROC AUC ===
model  baseline_rf  hierarchy_rf  transition_rf
fold                                           
0           0.5239        0.5285         0.5335
1           0.5019        0.5122         0.5187
2           0.5117        0.50

## Comparison with NB10 (50/50 split) and NB11 (50/50 split, fold-shift)


In [7]:
print("Baseline RF IC mean ± std:")
print(f"  NB10 (50/50 split):  +0.020 ± 0.011")
print(f"  NB11 (50/50 split):  +0.020 ± 0.021  (1h pulled from 3-horizon notebook)")
print(f"  NB08/NB09 (full train, no Stage 1):  ~+0.027")
print(f"  NB12 (OOF, full train):  {reg[reg['model']=='baseline_rf']['ic'].mean():+.5f} ± {reg[reg['model']=='baseline_rf']['ic'].std():.5f}")
print()
print("Hierarchy RF IC mean ± std:")
print(f"  NB10 (50/50 split):  +0.028 ± 0.013")
print(f"  NB11 (50/50 split):  +0.022 ± 0.019  (1h horizon)")
print(f"  NB12 (OOF, full train):  {reg[reg['model']=='hierarchy_rf']['ic'].mean():+.5f} ± {reg[reg['model']=='hierarchy_rf']['ic'].std():.5f}")
print()
print("Transition RF IC mean ± std:")
print(f"  NB10 (50/50 split):  +0.028 ± 0.007")
print(f"  NB11 (50/50 split):  +0.032 ± 0.020  (1h horizon)")
print(f"  NB12 (OOF, full train):  {reg[reg['model']=='transition_rf']['ic'].mean():+.5f} ± {reg[reg['model']=='transition_rf']['ic'].std():.5f}")


Baseline RF IC mean ± std:
  NB10 (50/50 split):  +0.020 ± 0.011
  NB11 (50/50 split):  +0.020 ± 0.021  (1h pulled from 3-horizon notebook)
  NB08/NB09 (full train, no Stage 1):  ~+0.027
  NB12 (OOF, full train):  +0.02599 ± 0.02643

Hierarchy RF IC mean ± std:
  NB10 (50/50 split):  +0.028 ± 0.013
  NB11 (50/50 split):  +0.022 ± 0.019  (1h horizon)
  NB12 (OOF, full train):  +0.03176 ± 0.02565

Transition RF IC mean ± std:
  NB10 (50/50 split):  +0.028 ± 0.007
  NB11 (50/50 split):  +0.032 ± 0.020  (1h horizon)
  NB12 (OOF, full train):  +0.03278 ± 0.02404


## Append to EXPERIMENTS.md


In [8]:
def append_to_journal(agg_df, info_df, ic_pivot, ba_pivot, auc_pivot, journal_path=JOURNAL_PATH):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    config = {
        "notebook": "12_two_stage_kfold_oof_1h.ipynb",
        "ticker": TICKER, "period": PERIOD, "interval": INTERVAL,
        "horizon_hours": HORIZON, "n_folds": N_FOLDS,
        "inner_n_splits": INNER_N_SPLITS,
        "max_hierarchy_depth": MAX_HIERARCHY_DEPTH,
        "stage1_params": STAGE1_PARAMS, "stage2_params": STAGE2_PARAMS,
        "rf_n_estimators": RF_N_ESTIMATORS, "seed": SEED,
    }
    reg_summary = agg_df[agg_df["kind"] == "regression"].groupby("model")[["mae", "rmse", "ic"]].agg(["mean", "std"]).round(5)
    cls_summary = agg_df[agg_df["kind"] == "classification"].groupby("model")[["accuracy", "balanced_accuracy", "roc_auc"]].agg(["mean", "std"]).round(5)
    ic_lift = (ic_pivot["transition_rf"] - ic_pivot["hierarchy_rf"]).round(5)
    ba_lift = ((ba_pivot["transition_rf"] - ba_pivot["hierarchy_rf"]) * 100).round(3)
    auc_lift = (auc_pivot["transition_rf"] - auc_pivot["hierarchy_rf"]).round(5)
    lift_df = pd.DataFrame({"ic_lift": ic_lift, "bal_acc_lift_pp": ba_lift, "auc_lift": auc_lift})

    parts = ["\n---\n"]
    parts.append(f"\n## {timestamp} — NB12 two-stage with K-fold OOF cross-fitting (BTC-USD, 1h, K=4)\n")
    parts.append(f"\n**Config**\n\n```json\n{json.dumps(config, indent=2)}\n```\n")
    parts.append("\n**Per-fold lifts: `transition_rf − hierarchy_rf`**\n\n")
    parts.append(lift_df.to_markdown())
    parts.append("\n\n**Per-fold info (OOF coverage, Stage-1 sanity)**\n\n")
    parts.append("```\n"); parts.append(str(info_df.round(4))); parts.append("\n```\n")
    parts.append("\n**Regression summary (mean ± std across folds)**\n\n")
    parts.append("```\n"); parts.append(str(reg_summary)); parts.append("\n```\n")
    parts.append("\n**Classification summary (mean ± std across folds)**\n\n")
    parts.append("```\n"); parts.append(str(cls_summary)); parts.append("\n```\n")
    text = "".join(parts)
    with open(journal_path, "a") as f:
        f.write(text)
    print(f"Appended entry to {journal_path}")
    return text


journal_text = append_to_journal(agg, info_df, ic_pivot, ba_pivot, auc_pivot)


Appended entry to EXPERIMENTS.md
